In [43]:
import torch 

def mtr_gen(m, n, r):
    core = torch.zeros((m, n))
    core[:r, :r] = torch.eye(r)
    return torch.randn((m, m)) @ core @ torch.randn((n, n))

In [57]:
import torch
import torch.nn as nn
from functools import partial

class TMP:
    def _w2d_converter(func):
        def __wrapping(self, W: torch.Tensor, *args, **kwargs):
            if not W.ndim == 2:
                W = self._flatten(W)
            result = func(self, W, *args, **kwargs)
            return result
        return __wrapping
    
    _flatten_funcs = {
        None: lambda x: x,
        'start_1': partial(torch.flatten, start_dim=1)
    }
    _agg_funcs = {
        'sum': torch.sum, 
        'mean': torch.mean,
        'max': torch.max
    }

    def __init__(self, flatten_mode: str = None, ):
        self.flatten_mode = flatten_mode

    def _flatten(self, W) -> torch.Tensor:
        W = self._flatten_funcs[self.flatten_mode](W)
        assert W.ndim == 2
        return W

    @_w2d_converter
    def get_importances(self, W2d: torch.Tensor, aggr: str = 'sum'):
        assert W2d.ndim == 2
        U, S, V = torch.svd(W2d)
        U_ = U * S
        assert torch.allclose(U_ @ V.T, W2d, atol=1e-5)
        V_ = (S * V).T
        assert torch.allclose(U @ V_, W2d, atol=1e-5)
        R_imp = self._agg_funcs[aggr](U_, dim=1)
        C_imp = self._agg_funcs[aggr](V_, dim=0)
        assert torch.outer(R_imp, C_imp).size() == W2d.size()
        return R_imp, C_imp
    
    @_w2d_converter
    def sample_indices(self, W, mode='topk', k=1):
        R_imp, C_imp = self.get_importances(W)
        if mode == 'topk':
            assert k >= 1
            C_ind = torch.sort(C_imp, descending=True).indices[:k]
            R_ind = torch.sort(R_imp, descending=True).indices[:k]
        return R_ind, C_ind
    
    @_w2d_converter
    def form_decomposition(self, W, k=10):
        R_ind, C_ind = self.sample_indices(W, k=k)
        return (
            W[:, C_ind], 
            torch.linalg.inv(W[R_ind][:, C_ind]),
              W[R_ind, :] 
        )
    
    @staticmethod
    def check_approximation_quality(C, U, R, W):
        return (C @ U @ R - W).abs().max() / W.abs().max()

    @staticmethod
    def criterion(U):
        assert U.size(0) == U.size(1)
        return torch.linalg.matrix_rank(U) == U.size(0)
# Tests
TMP().get_importances(torch.randn((100, 500)))
TMP().get_importances(torch.randn((100, 50)))

X = mtr_gen(100, 50, 10)
CUR = TMP('start_1').form_decomposition(X, 10)


In [58]:
TMP('start_1').criterion(CUR[1])

tensor(True)

In [59]:
TMP.check_approximation_quality(*CUR, X)

tensor(9.2499e-05)